In [1]:
import torch
import matplotlib.pyplot as plt
import numpy as np

from pytorch_ood.detector import KNN, Mahalanobis, GMM
from pytorch_ood.utils import OODMetrics

In [2]:
import torch


medsam_zs_shifts={
    "id": "MedSAM_zs_ZGT_masses",
    "near": "medSAM_zs_INbreast_masses",
    "far": "medSAM_zs_MamaMia_masses",
}


medsam_lora_shifts={
    "id": "medSAM_LoRA_ZGT_allmasses",
    "near": "medSAM_LoRA_INbreast_allmasses",
    "far": "medSAM_LoRA_MamaMia_masses",
}


versamammo_shifts = {
    "id": "VersaMammo_ZGT_allmasses",
    "near": "VersaMammo_INbreast_allmasses",
    "far": "VersaMammo_MamaMia_masses",
}

mammofm_shifts = {
    "id": "MammoFM_ZGT_allmasses",
    "near": "MammoFM_INbreast_allmasses",
    "far": "MammoFM_MamaMia_masses",
}

detector_classes = {
    "Mahalanobis": Mahalanobis,
    "KNN": KNN,
    "GMM": GMM,
}

results = {
    "MedSAM_zs":{},
    "MedSAM_LoRA":{},
    "VersaMammo": {},
    "MammoFM": {},
}

model_shifts = {
    "MedSAM_zs":medsam_zs_shifts,
    "MedSAM_LoRA":medsam_lora_shifts,
    "VersaMammo": versamammo_shifts,
    "MammoFM": mammofm_shifts,
}

for model_name, shifts in model_shifts.items():
    # Load each feature set once for this model
    
    Z_id = torch.load(shifts["id"], map_location="cpu")
    
    N_fit = int(Z_id.shape[0] * 0.8)
    Z_id_test = Z_id[N_fit:, :]
    Z_id_fit = Z_id[:N_fit, :]

    Z_near_ood = torch.load(shifts["near"], map_location="cpu")
    Z_far_ood = torch.load(shifts["far"], map_location="cpu")

    Y_id = torch.zeros(Z_id_fit.shape[0], dtype=torch.long)

    for detector_name, DetectorCls in detector_classes.items():
        det = DetectorCls(encoder=None)
        det.fit_features(Z_id_fit, Y_id)

        scores_id=  det.predict_features(Z_id_test)
        scores_near_ood = det.predict_features(Z_near_ood)
        scores_far_ood = det.predict_features(Z_far_ood)

        # Ensure outputs are tensors, since some detectors may return NumPy arrays
        scores_id = torch.as_tensor(
            scores_id,
            dtype=torch.float32,
        ).flatten()
        
        scores_near_ood = torch.as_tensor(
            scores_near_ood,
            dtype=torch.float32,
        ).flatten()

        scores_far_ood = torch.as_tensor(
            scores_far_ood,
            dtype=torch.float32,
        ).flatten()

        # Joint normalization makes near/far distances comparable
        all_scores = torch.cat(
            [scores_id, scores_near_ood, scores_far_ood],
            dim=0,
        )

        score_min = all_scores.min()
        score_max = all_scores.max()
        score_range = score_max - score_min

        if score_range.item() > 0:
            
            scores_id_norm = (
                scores_id - score_min
            ) / score_range            
            
            
            scores_near_ood_norm = (
                scores_near_ood - score_min
            ) / score_range

            scores_far_ood_norm = (
                scores_far_ood - score_min
            ) / score_range
        else:
            
            scores_id_norm = torch.zeros_like(scores_id)
            scores_near_ood_norm = torch.zeros_like(scores_near_ood)
            scores_far_ood_norm = torch.zeros_like(scores_far_ood)

        results[model_name][detector_name] = {
            
            "id_avg": scores_id_norm.mean().item(),
            "near_avg": scores_near_ood_norm.mean().item(),
            "far_avg": scores_far_ood_norm.mean().item(),
        }

In [3]:
from pprint import pprint

pprint(results)

{'MammoFM': {'GMM': {'far_avg': 0.5478372573852539,
                     'id_avg': 0.11850539594888687,
                     'near_avg': 0.4967266619205475},
             'KNN': {'far_avg': 0.7175861597061157,
                     'id_avg': 0.3814399540424347,
                     'near_avg': 0.6973494291305542},
             'Mahalanobis': {'far_avg': 0.6295809149742126,
                             'id_avg': 0.636685848236084,
                             'near_avg': 0.5654560923576355}},
 'MedSAM_LoRA': {'GMM': {'far_avg': 0.07765188813209534,
                         'id_avg': 0.001395705039612949,
                         'near_avg': 0.5319781303405762},
                 'KNN': {'far_avg': 0.4343971014022827,
                         'id_avg': 0.07317081093788147,
                         'near_avg': 0.8085793852806091},
                 'Mahalanobis': {'far_avg': 0.06278587877750397,
                                 'id_avg': 0.0015345874708145857,
                               

In [4]:
import json

with open("ood_average_normalized_distances.json", "w") as f:
    json.dump(results, f, indent=4)